# 🌐 Notebook 3: ETag and HMAC — hashes in the real world

A raw checksum is a building block. Two very common patterns sit on top of it:

1. **ETag** — an HTTP header that carries a hash of a resource so clients can ask *"has this changed?"* and servers can answer *"no, reuse your cached copy"*. Saves bandwidth, saves latency.
2. **HMAC** — a **keyed** hash. A plain SHA-256 detects accidental corruption but not a *smart* attacker (they can recompute the hash after changing the data). HMAC adds a secret key so only someone who knows the key can produce a valid tag.

This notebook shows both — no external services, all in-process.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/checksum
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🏷️ Part 1 — ETag and HTTP conditional GET

An **ETag** (entity tag) is usually a hash of the response body. Here is the flow:

1. First request: server returns the body + `ETag: "abc123"`.
2. Client caches it.
3. Next time, client sends `If-None-Match: "abc123"`.
4. Server computes the current ETag. If it matches → respond `304 Not Modified` with **no body**. Otherwise → `200` + new body + new ETag.

Used by: CDNs, browsers, S3 (`GetObject`), GitHub API, etc. It turns expensive reads into cheap *"did anything change?"* checks.

We will build a tiny in-memory version to see the mechanism end-to-end.

In [ ]:
import hashlib

def etag(body: bytes) -> str:
    # Real servers often use MD5, SHA-1, or a truncated hash. We use SHA-256 truncated to 16 hex chars.
    return '"' + hashlib.sha256(body).hexdigest()[:16] + '"'

class TinyServer:
    """Imagine this is your HTTP endpoint serving /profile.json"""
    def __init__(self, body: bytes):
        self.body = body

    def get(self, if_none_match: str | None = None):
        current = etag(self.body)
        if if_none_match == current:
            return {'status': 304, 'etag': current, 'body': None}
        return {'status': 200, 'etag': current, 'body': self.body}

server = TinyServer(b'{"user":"alice","plan":"pro"}')

# 1) First client request — no cache yet
r1 = server.get()
print('first GET        :', r1['status'], r1['etag'], r1['body'])
client_cache = {'etag': r1['etag'], 'body': r1['body']}

# 2) Client revalidates — sends If-None-Match
r2 = server.get(if_none_match=client_cache['etag'])
print('revalidate (hit) :', r2['status'], r2['etag'], '-- no body sent, save bandwidth')

# 3) Server data changes (user upgraded plan)
server.body = b'{"user":"alice","plan":"enterprise"}'
r3 = server.get(if_none_match=client_cache['etag'])
print('revalidate (miss):', r3['status'], r3['etag'], r3['body'])

assert r1['status'] == 200 and r1['body'] is not None
assert r2['status'] == 304 and r2['body'] is None, 'a hit must not resend the body'
assert r2['etag'] == r1['etag']
assert r3['status'] == 200 and r3['etag'] != r1['etag'], 'changed body must change the ETag'
print('\n✔ same bytes -> same ETag -> 304 with no body; different bytes -> new ETag')


**Takeaway:** ETag = checksum as a cache key. It is a tiny, beautiful optimization. Browsers and CDNs rely on it constantly.

Two common variants:
- **Strong ETag** (exact byte-equality, usually a hash of the body) — what we built.
- **Weak ETag** (prefixed `W/`, "semantically equivalent") — the server promises the response *means* the same thing, but bytes may differ (e.g., whitespace).

## 🔑 Part 2 — HMAC: tamper-proof checksums

A plain hash is not enough if someone **malicious** can touch the data *and* the checksum. They will just recompute the hash. Example: a signed URL you store in a user's cookie — if you just attach `sha256(payload)`, the user edits the payload and recomputes the hash, and your server has no way to tell.

**HMAC** (Hash-based Message Authentication Code) fixes this: the hash is keyed with a secret only the server knows.

- Server computes: `tag = HMAC(secret, payload)` and sends `(payload, tag)`.
- Attacker can change `payload` but cannot produce a new valid `tag` without the `secret`.
- Server verifies: recompute `HMAC(secret, payload)` and compare in **constant time**.

Used by: session cookies, JWT with `HS256`, AWS Signature V4 signed requests, webhook signatures (Stripe, GitHub), signed URLs for S3/CloudFront.

In [ ]:
import hmac, hashlib, secrets

SECRET = secrets.token_bytes(32)  # only the server knows this

def sign(payload: bytes) -> bytes:
    return hmac.new(SECRET, payload, hashlib.sha256).digest()

def verify(payload: bytes, tag: bytes) -> bool:
    expected = sign(payload)
    # compare_digest prevents timing attacks
    return hmac.compare_digest(expected, tag)

# Server issues a signed "cookie" to the user: (payload, tag)
payload = b'user=alice;role=user'
tag = sign(payload)
print('cookie:', payload, tag.hex()[:16], '...')

# Honest round-trip: server receives the cookie back and verifies
print('honest verify :', verify(payload, tag))

# Attacker tries to escalate privileges by editing the payload
tampered = b'user=alice;role=admin'
print('tampered      :', verify(tampered, tag))  # -> False, caught

# Can the attacker forge a tag? Not without the secret.
forged_tag = hashlib.sha256(tampered).digest()  # plain hash, no secret
print('forged tag    :', verify(tampered, forged_tag))  # -> False

assert verify(payload, tag) is True
assert verify(tampered, tag) is False, 'tampered payload must not verify'
assert verify(tampered, forged_tag) is False, 'a plain hash is not a valid tag'
# This is the exact attack that walked straight through the unkeyed scheme in
# notebook 2: rewrite the payload, recompute the digest. Here it fails, and the
# only thing that changed is that the server holds a key the attacker does not.
assert verify(payload, sign(payload)) is True
print('\n✔ the notebook-2 forgery no longer works — the key is the whole difference')


### Pitfalls worth remembering

- **Never compare digests with `==`.** Use `hmac.compare_digest` — it runs in constant time so an attacker can't learn the tag byte-by-byte from timing.
- **Don't roll your own `hash(secret + payload)`.** It is vulnerable to *length-extension* attacks with SHA-1/SHA-256. That's literally what HMAC was invented to fix — use it.
- **A hash is not encryption.** HMAC protects *integrity*, not *secrecy*. The payload is still readable. If you also need secrecy, combine with encryption (or use AEAD like AES-GCM).
- **Rotate keys.** Have a way to change the secret without invalidating everything at once (e.g., accept the old key during a grace window).

## 📌 Recap of the whole lab

1. **Notebook 1**: without a checksum, a single flipped bit can silently corrupt your data and all its replicas.
2. **Notebook 2**: a short hash stored next to the data detects corruption on read. Choose the algorithm based on threat model (random errors vs active attacker) and performance.
3. **Notebook 3**: the same primitive powers two huge real-world patterns — **ETag** (cache
   revalidation) and **HMAC** (authenticated messages).

### The one-line version

| Question | Tool |
|---|---|
| Did these bytes change by accident? | checksum (CRC32 for hardware faults, SHA-256 if it also names things) |
| Did *someone* change these bytes? | **MAC** — HMAC with a shared secret |
| Did someone change these bytes, and can I prove *who* wrote them to a third party? | digital signature (Ed25519, RSA) — a MAC can't do this, since both sides hold the same key |
| Are these bytes secret? | none of the above — that's encryption |

Next stop in this repo: `merkle-trees/` — what happens when you need to checksum *billions* of items and diff two replicas efficiently.